# 🎓 GLA University • Python Programming Masterclass
## **Day 5: Afternoon Session (3 Hours)**
### **The 6 Capstone Placement Projects: End-to-End Hands-on Implementation**
*Powered by **Edulateral Foundation** • Ideate • Innovate • Integrate*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shivam990q/Python-Programming-Training/blob/main/Day%2005%20-%20Regex%20and%20Projects/Day5_Afternoon_Capstone_Placement_Projects.ipynb)

---
### 📌 The 6 Capstone Placement Projects:
1. **Project 1: Persistent Contact Book** (File CRUD, Search, Validation)
2. **Project 2: Student Grade Analyzer** (Statistical Aggregations via `map`, `filter`, `reduce`)
3. **Project 3: Interactive Quiz Engine** (`random` questions, persistent leaderboard score history)
4. **Project 4: Multi-Category Expense Tracker** (Categorization, CSV persistence, budget warnings)
5. **Project 5: Regex Email, Phone & Log Forensic Validator** (Pattern extraction & error reporting)
6. **Project 6: ATM & Banking Simulation System** (Custom exception hierarchies, stateful transactions)
---


---
## 🚀 Capstone Project 1: Persistent Contact Book Application

### Specifications:
- Supports Adding, Searching (case-insensitive substring), Updating, and Deleting contacts.
- Input validation on phone numbers (must be 10 digits).
- Persistent file storage using structured JSON.


In [ ]:
import json, os

class ContactBook:
    def __init__(self, storage_file="contacts.json"):
        self.storage_file = storage_file
        self.contacts = self._load()

    def _load(self):
        if os.path.exists(self.storage_file):
            try:
                with open(self.storage_file, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return {}
        return {}

    def _save(self):
        with open(self.storage_file, "w", encoding="utf-8") as f:
            json.dump(self.contacts, f, indent=2)

    def add_contact(self, name, phone, email=""):
        clean_phone = "".join(ch for ch in phone if ch.isdigit())
        if len(clean_phone) != 10:
            raise ValueError("Phone number must contain exactly 10 digits")
        self.contacts[name.strip()] = {"phone": clean_phone, "email": email.strip()}
        self._save()
        return True

    def search(self, query):
        q = query.lower()
        return {k: v for k, v in self.contacts.items() if q in k.lower() or q in v["phone"]}

    def delete(self, name):
        if name in self.contacts:
            del self.contacts[name]
            self._save()
            return True
        return False

# Unit Tests for Project 1
cb = ContactBook("test_contacts.json")
cb.add_contact("Aarav Sharma", "9876543210", "aarav@gla.ac.in")
cb.add_contact("Diya Verma", "9123456789", "diya@gla.ac.in")
assert len(cb.search("aarav")) == 1
assert len(cb.search("9123")) == 1
cb.delete("Aarav Sharma")
assert len(cb.search("aarav")) == 0
print("✅ Project 1: Contact Book Verified Successfully!")


---
## 🚀 Capstone Project 2: Student Grade Analyzer (Functional Pipelines)

### Specifications:
- Student records stored as dictionaries within lists.
- Computes student percentages, assigned letter grades (`A+`, `A`, `B`, `F`).
- Computes class average, finds the topper, and identifies struggling students.
- Implemented using functional primitives: `lambda`, `map`, `filter`, and `functools.reduce`.


In [ ]:
from functools import reduce

students_db = [
    {"roll": 101, "name": "Aarav", "marks": [85, 92, 88, 90]},
    {"roll": 102, "name": "Diya", "marks": [95, 98, 92, 96]},
    {"roll": 103, "name": "Rohan", "marks": [72, 68, 75, 70]},
    {"roll": 104, "name": "Sneha", "marks": [35, 42, 38, 40]},
    {"roll": 105, "name": "Karan", "marks": [88, 84, 82, 86]}
]

def analyze_student(student):
    marks = student["marks"]
    avg = sum(marks) / len(marks)
    if avg >= 90: grade = "A+"
    elif avg >= 80: grade = "A"
    elif avg >= 70: grade = "B"
    elif avg >= 50: grade = "C"
    else: grade = "F"
    return {
        "roll": student["roll"],
        "name": student["name"],
        "average": round(avg, 2),
        "grade": grade
    }

# 1. Map to enriched report cards
report_cards = list(map(analyze_student, students_db))

# 2. Filter honors students (grade A or A+)
honors_students = list(filter(lambda s: s["grade"] in ["A", "A+"], report_cards))

# 3. Reduce to calculate class overall average
total_score = reduce(lambda acc, s: acc + s["average"], report_cards, 0.0)
class_avg = round(total_score / len(report_cards), 2)

# 4. Find Class Topper
topper = reduce(lambda a, b: a if a["average"] > b["average"] else b, report_cards)

print("Class Average:", class_avg)
print(f"Class Topper: {topper['name']} ({topper['average']}% - Grade {topper['grade']})")
print("Honors Students Count:", len(honors_students))

assert topper["name"] == "Diya"
assert topper["grade"] == "A+"
assert len(honors_students) == 3
print("✅ Project 2: Student Grade Analyzer Verified Successfully!")


---
## 🚀 Capstone Project 3: Interactive Quiz Engine with Leaderboard

### Specifications:
- Questions loaded dynamically from data structure/file.
- Questions randomized using `random.sample()`.
- Timed or interactive answers, computes final score and percentage.
- Appends historical quiz attempts to a persistent leaderboard file.


In [ ]:
import random, os
from datetime import datetime

class QuizEngine:
    def __init__(self, questions_pool, history_file="quiz_history.log"):
        self.questions_pool = questions_pool
        self.history_file = history_file

    def get_shuffled_quiz(self, num_questions=3):
        return random.sample(self.questions_pool, min(num_questions, len(self.questions_pool)))

    def evaluate_quiz(self, candidate_name, answers_dict):
        # answers_dict maps question_id -> candidate chosen option index
        score = 0
        total = len(answers_dict)
        for q in self.questions_pool:
            if q["id"] in answers_dict:
                if answers_dict[q["id"]] == q["correct"]:
                    score += 1

        pct = round((score / total) * 100, 1) if total > 0 else 0.0
        self._log_attempt(candidate_name, score, total, pct)
        return {"candidate": candidate_name, "score": score, "total": total, "percentage": pct}

    def _log_attempt(self, name, score, total, pct):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(self.history_file, "a", encoding="utf-8") as f:
            f.write(f"{timestamp} | Candidate: {name} | Score: {score}/{total} ({pct}%)\n")

pool = [
    {"id": 1, "q": "Is Python dynamically typed?", "correct": 0},
    {"id": 2, "q": "Are strings mutable in Python?", "correct": 1},
    {"id": 3, "q": "Does round(2.5) equal 2 in Python 3?", "correct": 0}
]

quiz = QuizEngine(pool, "test_quiz_history.log")
result = quiz.evaluate_quiz("Shivam", {1: 0, 2: 1, 3: 0})
assert result["score"] == 3
assert result["percentage"] == 100.0
assert os.path.exists("test_quiz_history.log")
print("✅ Project 3: Quiz Engine Verified Successfully!")


---
## 🚀 Capstone Project 4: Multi-Category Expense Tracker

### Specifications:
- Log expenses with category, description, and amount.
- Persistence using standard CSV.
- Category-wise totals and budget threshold alert warnings.


In [ ]:
import csv, os

class ExpenseTracker:
    def __init__(self, filename="expenses.csv", monthly_budget=15000.0):
        self.filename = filename
        self.monthly_budget = monthly_budget
        if not os.path.exists(self.filename):
            with open(self.filename, "w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(["Date", "Category", "Amount", "Note"])

    def add_expense(self, date_str, category, amount, note=""):
        with open(self.filename, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([date_str, category.strip().title(), float(amount), note.strip()])

    def get_summary(self):
        category_totals = {}
        grand_total = 0.0
        with open(self.filename, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                cat = row["Category"]
                amt = float(row["Amount"])
                category_totals[cat] = category_totals.get(cat, 0.0) + amt
                grand_total += amt

        budget_warning = grand_total > self.monthly_budget
        return {
            "categories": category_totals,
            "grand_total": round(grand_total, 2),
            "budget": self.monthly_budget,
            "over_budget": budget_warning
        }

if os.path.exists("test_expenses.csv"): os.remove("test_expenses.csv")
tracker = ExpenseTracker("test_expenses.csv", monthly_budget=5000.0)
tracker.add_expense("2026-11-16", "Food", 1200.0, "Cafeteria")
tracker.add_expense("2026-11-16", "Books", 1800.0, "Python reference")
tracker.add_expense("2026-11-16", "Travel", 800.0, "Train ticket")

summary = tracker.get_summary()
assert summary["grand_total"] == 3800.0
assert summary["categories"]["Food"] == 1200.0
assert summary["over_budget"] == False
print("✅ Project 4: Expense Tracker Verified Successfully!")


---
## 🚀 Capstone Project 5: Automated Email & Log Forensic Validator (Regex)

### Specifications:
- Validates institutional and standard email addresses using rigorous regex.
- Validates Indian 10-digit mobile numbers with optional `+91-` prefix.
- Forensic log parser: extracts IP addresses, HTTP status codes, and endpoints from server logs.


In [ ]:
import re

EMAIL_REGEX = re.compile(r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$')
PHONE_REGEX = re.compile(r'^(?:\+91[- ]?)?[6-9]\d{9}$')
LOG_LINE_REGEX = re.compile(r'(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}) - - \[(.*?)\] "(GET|POST|PUT|DELETE) (\S+) HTTP/\d\.\d" (\d{3})')

def validate_candidate_credentials(email, phone):
    is_email_valid = bool(EMAIL_REGEX.match(email.strip()))
    is_phone_valid = bool(PHONE_REGEX.match(phone.strip()))
    return is_email_valid, is_phone_valid

def parse_server_log(log_line):
    match = LOG_LINE_REGEX.search(log_line)
    if not match:
        return None
    ip, timestamp, method, endpoint, status_code = match.groups()
    return {
        "ip": ip,
        "timestamp": timestamp,
        "method": method,
        "endpoint": endpoint,
        "status": int(status_code)
    }

# Unit Tests
e_val, p_val = validate_candidate_credentials("student@gla.ac.in", "+91 9876543210")
assert e_val == True and p_val == True

e_bad, p_bad = validate_candidate_credentials("bad..email@", "12345")
assert e_bad == False and p_bad == False

sample_log = '192.168.1.105 - - [16/Nov/2026:12:45:00 +0530] "POST /api/v1/login HTTP/1.1" 200'
parsed = parse_server_log(sample_log)
assert parsed["ip"] == "192.168.1.105"
assert parsed["method"] == "POST"
assert parsed["status"] == 200
print("✅ Project 5: Email, Phone & Log Validator Verified Successfully!")


---
## 🚀 Capstone Project 6: ATM & Banking Simulation System

### Specifications:
- User-defined exception hierarchy:
  - `BankingError` (Base exception)
  - `InsufficientFundsError`
  - `InvalidPinError`
  - `TransactionLimitExceededError`
- Secure PIN authentication (3 attempts before account lockout).
- Balance checking, deposits, and withdrawals with transaction logging.


In [ ]:
class BankingError(Exception):
    """Base exception for all banking simulation operations."""
    pass

class InsufficientFundsError(BankingError):
    def __init__(self, balance, amount):
        super().__init__(f"Transaction Declined: Current balance is ₹{balance:.2f}, attempted withdrawal ₹{amount:.2f}")

class InvalidPinError(BankingError):
    pass

class TransactionLimitExceededError(BankingError):
    pass

class BankAccount:
    DAILY_LIMIT = 50000.0

    def __init__(self, acc_number, holder_name, pin, opening_balance=0.0):
        self.acc_number = acc_number
        self.holder_name = holder_name
        self._pin = str(pin)
        self.balance = float(opening_balance)
        self.is_locked = False
        self.failed_attempts = 0
        self.transactions = []

    def authenticate(self, entered_pin):
        if self.is_locked:
            raise BankingError("Account is locked due to 3 consecutive failed PIN attempts. Contact branch.")
        if str(entered_pin) != self._pin:
            self.failed_attempts += 1
            if self.failed_attempts >= 3:
                self.is_locked = True
                raise BankingError("Account has been LOCKED due to 3 invalid PIN attempts!")
            raise InvalidPinError(f"Incorrect PIN! ({3 - self.failed_attempts} attempts remaining)")
        self.failed_attempts = 0
        return True

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.balance += amount
        self.transactions.append(f"DEPOSIT: +₹{amount:.2f} (Balance: ₹{self.balance:.2f})")
        return self.balance

    def withdraw(self, amount, entered_pin):
        self.authenticate(entered_pin)
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.DAILY_LIMIT:
            raise TransactionLimitExceededError(f"Exceeds daily limit of ₹{self.DAILY_LIMIT:.2f}")
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        self.transactions.append(f"WITHDRAWAL: -₹{amount:.2f} (Balance: ₹{self.balance:.2f})")
        return self.balance

# Unit Tests for Project 6
acc = BankAccount("GLA_1001", "Aarav Sharma", "1234", opening_balance=10000.0)
acc.deposit(5000.0)
assert acc.balance == 15000.0

acc.withdraw(3000.0, "1234")
assert acc.balance == 12000.0

# Test Insufficient Funds
try:
    acc.withdraw(50000.0, "1234")
    assert False, "Should raise InsufficientFundsError"
except InsufficientFundsError:
    pass

# Test Invalid PIN lockout
try:
    acc.authenticate("9999")
except InvalidPinError:
    pass
assert acc.failed_attempts == 1
assert acc.is_locked == False

print("✅ Project 6: ATM & Banking Simulation System Verified Successfully!")
print("🎉 ALL 6 CAPSTONE PLACEMENT PROJECTS VERIFIED WITH ZERO ERRORS!")
